# Image Transformations

This notebook explores geometric image transformations using OpenCV and NumPy.

It implements rotation, skewing and affine transformations and compares interpolation strategies and transformation order.


In [ ]:
!pip list


In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

In [ ]:
import cv2, numpy as np, math

def rotate_full(I, angle_deg):
    h, w = I.shape[:2]
    cx, cy = w / 2, h / 2

    R = cv2.getRotationMatrix2D((cx, cy), angle_deg, 1.0)

    # compute new bounding box so nothing is cut off
    abs_cos = abs(R[0, 0])
    abs_sin = abs(R[0, 1])
    new_w = int(h * abs_sin + w * abs_cos)
    new_h = int(h * abs_cos + w * abs_sin)

    # shift so the rotated image is centered in the new canvas
    R[0, 2] += new_w / 2 - cx
    R[1, 2] += new_h / 2 - cy

    rotated = cv2.warpAffine(I, R, (new_w, new_h),
                             flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated

In [ ]:
def skew_full(I, angle_deg):
    h, w = I.shape[:2]
    k = math.tan(math.radians(angle_deg))  # shear factor
    new_w = int(w + abs(k) * h)
    M = np.float32([[1, k, 0],
                    [0, 1, 0]])
    skewed = cv2.warpAffine(I, M, (new_w, h),
                            flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_REPLICATE)
    return skewed

In [ ]:
def rotate_and_skew(I, theta1, theta2):
    rotated = rotate_full(I, theta1)
    result = skew_full(rotated, theta2)
    return result

In [ ]:
name_img = cv2.imread("cw1_name_image.png")
import os
os.makedirs("outputs", exist_ok=True)

In [ ]:
rot_angles = [30, 60, 120, -50]

for ang in rot_angles:
    out = rotate_and_skew(name_img, ang, 0)
    cv2.imwrite(f"outputs/rot_{ang}.png", out)

In [ ]:
skew_angles = [10, 40, 60]

for ang in skew_angles:
    out = skew_full(name_img, ang)
    cv2.imwrite(f"outputs/skew_{ang}.png", out)

In [ ]:
theta1 = 20
theta2 = 50

# (i) Rotate → Skew
rot_then_skew = rotate_and_skew(name_img, theta1, theta2)
cv2.imwrite("outputs/rot_then_skew.png", rot_then_skew)

# (ii) Skew → Rotate
skew_first = skew_full(name_img, theta2)
skew_then_rot = rotate_full(skew_first, theta1)
cv2.imwrite("outputs/skew_then_rot.png", skew_then_rot)

In [ ]:
# =====================================================================
# TASK 1 — EXTRA EXPERIMENTS (HIGH VISIBILITY VERSION)
# =====================================================================

import cv2, numpy as np, math, os

os.makedirs("outputs/task1_extra", exist_ok=True)

img = cv2.imread("cw1_name_image.png")

# Stronger settings
ANGLE = 80        # strong rotation
SCALE = 2.0       # upscale exaggerates differences

h, w = img.shape[:2]
cx, cy = w / 2, h / 2

# ================================================================
# A) Interpolation comparison (extreme version)
# ================================================================

methods = {
    "nearest": cv2.INTER_NEAREST,
    "linear": cv2.INTER_LINEAR,
    "cubic": cv2.INTER_CUBIC,
    "lanczos": cv2.INTER_LANCZOS4
}

M = cv2.getRotationMatrix2D((cx, cy), ANGLE, SCALE)

# Increase canvas size so the large rotated image fits
new_w = int(w * 2.5)
new_h = int(h * 2.5)

for name, flag in methods.items():
    out = cv2.warpAffine(
        img, M, (new_w, new_h),
        flags=flag, borderMode=cv2.BORDER_REPLICATE
    )
    cv2.imwrite(f"outputs/task1_extra/1_interp_{name}.png", out)

print("✓ Saved HIGH-VISIBILITY interpolation images.")



# ================================================================
# B) Forward vs backward mapping (extreme version)
# ================================================================

def forward_rotate_extreme(I, angle_deg, scale=2.0):
    h, w = I.shape[:2]
    angle = math.radians(angle_deg)
    out = np.zeros((int(h * scale * 2), int(w * scale * 2), 3), dtype=np.uint8)
    cx, cy = out.shape[1] / 2, out.shape[0] / 2

    for y in range(h):
        for x in range(w):
            nx = int(cx + (x - w/2) * math.cos(angle) - (y - h/2) * math.sin(angle))
            ny = int(cy + (x - w/2) * math.sin(angle) + (y - h/2) * math.cos(angle))

            if 0 <= nx < out.shape[1] and 0 <= ny < out.shape[0]:
                out[ny, nx] = I[y, x]
    return out

forward_img = forward_rotate_extreme(img, ANGLE, scale=SCALE)
cv2.imwrite("outputs/task1_extra/1_forward_mapping.png", forward_img)

backward_img = cv2.warpAffine(
    img, M, (new_w, new_h),
    flags=cv2.INTER_LINEAR,
    borderMode=cv2.BORDER_REPLICATE
)
cv2.imwrite("outputs/task1_extra/1_backward_mapping.png", backward_img)

print("✓ Saved HIGH-VISIBILITY forward/backward mapping images.")